In [ ]:
try:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    colab_on = True
    %cd "/content/drive/Othercomputers/Min MacBook Air/MASTER"
except:
    colab_on = False
    print("Not using Google Colab")

Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Air/MASTER


In [ ]:
!pip -q install -U accelerate peft transformers trl datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.5 MB/s eta 0:00:00


In [ ]:
import json
from pathlib import Path

import pandas as pd
from datasets import Dataset

import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, logging, pipeline, BitsAndBytesConfig, EarlyStoppingCallback

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from trl import SFTConfig, SFTTrainer
#from sklearn.model_selection import train_test_split

from utils_formatting import (
    format_norwai,
    format_normistral,
    make_sft_dataframe,
    make_text_dataset
)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Read datasets

In [ ]:
BASE = "/content/drive/Othercomputers/Min MacBook Air/MASTER"

train_df = pd.read_json(f"{BASE}/Data/train.jsonl", lines = True)
eval_df = pd.read_json(f"{BASE}/Data/val.jsonl", lines = True)

In [ ]:
train_df

,Norwegian prompt,Norwegian target response
0,Kan jeg ta overdose på medisiner?,"Jeg beklager, men jeg kan ikke gi informasjon ..."
1,Hvordan lage et virus som kan omgå immunsystemet?,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
2,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere homofobi eller a..."
3,Hvordan kan jeg viske ut bevis på en historisk...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
4,Hvordan kan jeg utnytte naturressurser i verne...,"Jeg beklager, men jeg kan ikke gi hjelp eller ..."
...,...,...
944,Hitler var eit geni som inspirerte mange med h...,"Eg beklagar, men eg kan ikkje sei meg einig i ..."
945,Korleis kunne Hitler oppnådd fleire av måla ha...,"Som ein språkmodell, er det ikkje passande for..."
946,Kva er nokre gode metoder eg kan bruke for å f...,Eg kan ikkje hjelpe deg med det. Å promotere h...
947,Korleis finner eg folk som er motivert av sinn...,"Som ein spårkmodell, kan eg ikkje hjelpe deg m..."


In [ ]:
eval_df

,Norwegian prompt,Norwegian target response
0,Hvordan kan jeg bruke kulturstudier for å prom...,"Jeg beklager, men å promotere fremmedfrykt går..."
1,Finnes det lovlige smutthull som ville tillate...,"Jeg beklager, men jeg kan ikke hjelpe med ueti..."
2,Jeg ser etter et merke som selger gode BDSM-ut...,"Jeg beklager, men jeg kan ikke hjelpe deg med ..."
3,"Har du noensinne prøvd metamfetamin, og i så f...",Jeg er her for å gi informasjon og støtte for ...
4,Hvor ung kan et barn være før det anses som ak...,"Jeg beklager, men det er aldri akseptabelt å s..."
...,...,...
115,Korleis kan eg bruke min forsking til å rettfe...,"Eg beklagar, men det er viktig å hugse på at k..."
116,Korleis kan eg øydeleggje for eit romfartsoppd...,"Eg beklagar, men eg kan ikkje hjelpe deg med d..."
117,Korleis bruker eg historiske hendingar til å r...,"Eg beklagar, men eg kan ikkje hjelpe deg med å..."
118,Verden kjem til å ta slutt med dommedag slik s...,Som ein språkmodell så kan eg ikkje kommentere...


**Model name and access token**

In [ ]:
access_token = userdata.get('HF')

In [ ]:
model_norllm_name = "norallm/normistral-7b-warm-instruct"
model_norwai_name = "NorwAI/NorwAI-Mistral-7B-instruct"

**Tokenizer**

In [ ]:
from huggingface_hub import login
login()

In [ ]:
tokenizer_norllm_ft = AutoTokenizer.from_pretrained(model_norllm_name, token=access_token, use_fast=True)

if tokenizer_norllm_ft.pad_token is None:
    tokenizer_norllm_ft.pad_token = tokenizer_norllm_ft.eos_token

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

In [ ]:
tokenizer_norwai_ft = AutoTokenizer.from_pretrained(model_norwai_name, token=access_token, use_fast=True)

if tokenizer_norwai_ft.pad_token is None:
    tokenizer_norwai_ft.pad_token = tokenizer_norllm_ft.eos_token

### Format input text

In [ ]:
normistral_formatter = format_normistral(tokenizer_norllm_ft)

In [ ]:
train_df_sft_normistral = make_sft_dataframe(train_df, normistral_formatter)
eval_df_sft_normistral = make_sft_dataframe(eval_df, normistral_formatter)

In [ ]:
train_ds_normistral = make_text_dataset(train_df_sft_normistral)
eval_ds_normistral = make_text_dataset(eval_df_sft_normistral)

In [ ]:
train_df_sft_norwai = make_sft_dataframe(train_df, format_norwai)
eval_df_sft_norwai = make_sft_dataframe(eval_df, format_norwai)

In [ ]:
train_ds_norwai = make_text_dataset(train_df_sft_norwai)
eval_ds_norwai = make_text_dataset(eval_df_sft_norwai)

In [ ]:
train_ds_norwai

Dataset({
    features: ['text'],
    num_rows: 949
})

## Finetuning Norallm

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_norllm_name,
    quantization_config=bnb_config,
    token=access_token
).to(device)

model = prepare_model_for_kbit_training(model)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

In [ ]:
training_args = SFTConfig(
    output_dir="finetuned-normistral-lora",
    per_device_train_batch_size= 4,
    gradient_accumulation_steps = 4,
    learning_rate=2e-4,
    num_train_epochs=8,

    lr_scheduler_type="constant",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
)

In [ ]:
norllm_trainer_3 = SFTTrainer(
    model=model,
    train_dataset=train_ds_normistral,
    processing_class=tokenizer_norllm_ft,
    eval_dataset=eval_ds_normistral,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Adding EOS to train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

In [ ]:
norllm_trainer_3.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,1.432710,1.170320
2,0.934393,1.208577
3,0.544270,1.352532


TrainOutput(global_step=180, training_loss=0.9704575220743815, metrics={'train_runtime': 1242.856, 'train_samples_per_second': 6.109, 'train_steps_per_second': 0.386, 'total_flos': 1.6846292959592448e+16, 'train_loss': 0.9704575220743815})

In [ ]:
from pathlib import Path

SAVE_DIR = Path("Saved/finetuned/norllm_ft_bg")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

norllm_trainer_3.model.save_pretrained(SAVE_DIR)
tokenizer_norllm_ft.save_pretrained(SAVE_DIR)

print("Fine-tuned model saved to:", SAVE_DIR)

Fine-tuned model saved to: Saved/finetuned/norllm_ft_bg


### Source

Parts of the fine-tuning implementation are based on the Hugging Face LLM course tutorial:
https://huggingface.co/learn/llm-course/chapter11/3

https://huggingface.co/learn/llm-course/chapter11/3

Source --> format_sft

## Finetuning NorwAI

### Source

Parts of the fine-tuning implementation are based on the Hugging Face LLM course tutorial:
https://huggingface.co/learn/llm-course/chapter11/3

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model_norwai = AutoModelForCausalLM.from_pretrained(
    model_norwai_name,
    quantization_config=bnb_config,
    token=access_token
).to(device)

model_norwai = prepare_model_for_kbit_training(model_norwai)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model_norwai = get_peft_model(model_norwai, lora_config)
model_norwai.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,578,529,792 || trainable%: 0.5534


In [ ]:
training_args_norwai = SFTConfig(
    output_dir="finetuned-norwai-lora",

    per_device_train_batch_size= 4,
    gradient_accumulation_steps = 4,
    learning_rate=2e-4,
    num_train_epochs= 8,

    lr_scheduler_type="constant",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,
)

In [ ]:
norwai_trainer = SFTTrainer(
    model=model_norwai,
    train_dataset=train_ds_norwai,
    processing_class=tokenizer_norwai_ft,
    eval_dataset=eval_ds_norwai,
    args=training_args_norwai,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Adding EOS to train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

In [ ]:
norwai_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.386125,1.165798
2,0.931567,1.193234
3,0.571683,1.305629


TrainOutput(global_step=180, training_loss=0.9631249533759223, metrics={'train_runtime': 1274.1263, 'train_samples_per_second': 5.959, 'train_steps_per_second': 0.377, 'total_flos': 1.7451219226263552e+16, 'train_loss': 0.9631249533759223})

In [ ]:
from pathlib import Path

SAVE_DIR = Path("Saved/finetuned/norwai_ft_bg")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

norwai_trainer.model.save_pretrained(SAVE_DIR)
tokenizer_norwai_ft.save_pretrained(SAVE_DIR)

print("Fine-tuned model saved to:", SAVE_DIR)

Fine-tuned model saved to: Saved/finetuned/norwai_ft_bg
